# Does the architecture earn its place?

The manuscript names three design decisions and argues for each in prose, but none has ever
been ablated:

1. **The direct cardiorespiratory bypass.** The respiratory head reads the BiLSTM context
   *and* the raw per-epoch cardiorespiratory embedding, so desaturation reaches the decision
   without being squeezed through a fusion trained mostly for staging. The paper asserts
   "this direct connection matters" with no supporting experiment.
2. **The temporal decoder.** A BiLSTM over 20-epoch windows supplies night context. Its
   contribution has never been separated from the per-epoch features or from the HMM
   smoothing layered on top.
3. **Joint multi-task training.** Both heads share one encoder and one decoder and are
   trained under a summed loss. Whether that beats two independent single-task models — the
   obvious alternative, and the one a reader will ask about — has never been tested.

Each is a claim the paper makes about its own architecture, and each is answerable by an
experiment that takes minutes. This notebook runs them, across three seeds, with dispersion
and corrected tests.

A note on what a negative result would mean. If the bypass turns out not to matter, that is
worth knowing and worth reporting — but it is a different finding from "we never checked".
The point of this notebook is to replace an assertion with a measurement, whichever way it
falls.

In [1]:
import json
import os
import sys
import time

import numpy as np

REPO = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "..", ".."))
sys.path.insert(0, os.path.join(REPO, "MMNet_research", "model"))
import mmnet_core as C  # noqa: E402

OUT = os.path.join(REPO, "MMNet_research", "results", "revision", "runs")
CACHE = os.path.join(OUT, "architecture_ablation.json")
SEEDS = [42, 1, 7]

# name -> kwargs for run_10fold
ARCH = {
    "full (as published)":      dict(),
    "no cardio bypass":         dict(bypass=False),
    "no temporal decoder":      dict(temporal="none"),
    "single-task: staging":     dict(task="stage"),
    "single-task: respiratory": dict(task="apnea"),
}
print("conditions: %d | seeds: %s | runs: %d" % (len(ARCH), SEEDS, len(ARCH) * len(SEEDS)))
print("device:", C.DEV)

cwd: D:\sleep-staging-psg\MMNet_research\MMNet_Submission\all_codes\notebooks | device: cuda | NVIDIA GeForce RTX 2060


subjects: 96 (SN28 dropped) | epochs: 89,532
stage %: {'W': np.float64(26.6), 'N1': np.float64(10.2), 'N2': np.float64(42.3), 'N3': np.float64(8.9), 'R': np.float64(12.1)}
respiratory-event prevalence: 16.0%


EEG-feature counts -> EEG: 112 EOG: 50 EMG: 26
parameters (concat): 773,254
training utilities defined.
conditions: 5 | seeds: [42, 1, 7] | runs: 15
device: cuda


In [2]:
res = json.load(open(CACHE)) if os.path.exists(CACHE) else {}
t0 = time.time()
todo = [(n, s) for n in ARCH for s in SEEDS if "%s|%d" % (n, s) not in res]
print("%d runs remaining\n" % len(todo))

for name, seed in todo:
    r = C.run_10fold(fusion="concat", seed=seed, **ARCH[name])
    res["%s|%d" % (name, seed)] = {k: [f[k] for f in r["per_fold"]]
                                   for k in ("acc", "mf1", "kappa", "auc", "ap")}
    json.dump(res, open(CACHE, "w"))
    print("%-26s seed %-4d acc %.4f  AUC %.4f   (%.1f min)"
          % (name, seed, np.mean(res["%s|%d" % (name, seed)]["acc"]),
             np.nanmean(res["%s|%d" % (name, seed)]["auc"]), (time.time() - t0) / 60),
          flush=True)
print("\ntotal %.1f min" % ((time.time() - t0) / 60))

15 runs remaining



full (as published)        seed 42   acc 0.7227  AUC 0.7111   (9.8 min)


full (as published)        seed 1    acc 0.7280  AUC 0.7027   (18.6 min)


full (as published)        seed 7    acc 0.7208  AUC 0.7013   (24.8 min)


no cardio bypass           seed 42   acc 0.7246  AUC 0.7025   (31.3 min)


...

single-task: staging       seed 7    acc 0.7217  AUC 0.4191   (75.3 min)


single-task: respiratory   seed 42   acc 0.2939  AUC 0.7015   (80.2 min)


single-task: respiratory   seed 1    acc 0.1825  AUC 0.7028   (87.4 min)


single-task: respiratory   seed 7    acc 0.2140  AUC 0.6945   (93.6 min)



total 93.6 min


## Results, with the noise floor stated alongside

In [3]:
def pool(name, metric):
    return np.array([v for s in SEEDS if "%s|%d" % (name, s) in res
                     for v in res["%s|%d" % (name, s)][metric]])

print("%-26s %-22s %-22s" % ("condition", "staging acc", "respiratory AUC"))
print("-" * 72)
for n in ARCH:
    a, u = pool(n, "acc"), pool(n, "auc")
    sa = "%.4f +- %.3f" % (np.nanmean(a), np.nanstd(a)) if len(a) else "n/a"
    su = "%.4f +- %.3f" % (np.nanmean(u), np.nanstd(u)) if len(u) else "n/a"
    print("%-26s %-22s %-22s" % (n, sa, su))

condition                  staging acc            respiratory AUC       
------------------------------------------------------------------------
full (as published)        0.7238 +- 0.034        0.7050 +- 0.033       
no cardio bypass           0.7225 +- 0.027        0.6952 +- 0.041       
no temporal decoder        0.7386 +- 0.034        0.6767 +- 0.034       
single-task: staging       0.7224 +- 0.029        0.4835 +- 0.077       
single-task: respiratory   0.2301 +- 0.085        0.6996 +- 0.039       


In [4]:
from scipy.stats import ttest_rel, wilcoxon

def holm(p):
    p = np.asarray(p, float); order = np.argsort(p); m = len(p)
    adj = np.empty(m); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i]); adj[i] = min(1.0, run)
    return adj

def tost(d, margin):
    lo = ttest_rel(d, np.full_like(d, -margin)).pvalue / 2 if d.mean() > -margin else 1.0
    hi = ttest_rel(d, np.full_like(d, margin)).pvalue / 2 if d.mean() < margin else 1.0
    return max(lo, hi)

# each architecture variant is compared on the metric it is supposed to affect
TESTS = [("no cardio bypass",         "auc", "respiratory AUC"),
         ("no temporal decoder",      "acc", "staging accuracy"),
         ("no temporal decoder",      "auc", "respiratory AUC"),
         ("single-task: staging",     "acc", "staging accuracy"),
         ("single-task: respiratory", "auc", "respiratory AUC")]

raws, rows = [], []
for name, metric, label in TESTS:
    base, v = pool("full (as published)", metric), pool(name, metric)
    ok = ~(np.isnan(base) | np.isnan(v))
    d = v[ok] - base[ok]
    raws.append(wilcoxon(v[ok], base[ok]).pvalue)
    rows.append((name, label, d, float(np.nanstd(base[ok]))))
adj = holm(raws)

print("%-26s %-18s %9s %9s %9s  %s" % ("variant", "metric", "delta", "p Holm", "p TOST", "verdict"))
print("-" * 92)
for (name, label, d, sd), pr, pa in zip(rows, raws, adj):
    pt = tost(d, sd)
    if pa < 0.05:
        verdict = "COMPONENT EARNS ITS PLACE" if d.mean() < 0 else "REMOVING IT HELPS"
    elif pt < 0.05:
        verdict = "equivalent - component is redundant"
    else:
        verdict = "INCONCLUSIVE"
    print("%-26s %-18s %+9.4f %9.4f %9.4f  %s" % (name, label, d.mean(), pa, pt, verdict))

print("\ndelta is (variant - full). Negative means removing the component HURT,")
print("i.e. the component was doing real work.")

variant                    metric                 delta    p Holm    p TOST  verdict
--------------------------------------------------------------------------------------------
no cardio bypass           respiratory AUC      -0.0098    0.0244    0.0000  COMPONENT EARNS ITS PLACE
no temporal decoder        staging accuracy     +0.0147    0.0001    0.0000  REMOVING IT HELPS
no temporal decoder        respiratory AUC      -0.0283    0.0000    0.1772  COMPONENT EARNS ITS PLACE
single-task: staging       staging accuracy     -0.0014    0.6120    0.0000  equivalent - component is redundant
single-task: respiratory   respiratory AUC      -0.0054    0.3819    0.0000  equivalent - component is redundant

delta is (variant - full). Negative means removing the component HURT,
i.e. the component was doing real work.


## Reading the result

The row that matters most is **no cardio bypass** on respiratory AUC. That pathway is the
manuscript's named novel component; if removing it costs measurable AUC, the architectural
claim is established by experiment rather than asserted, and the contribution is a design
decision that demonstrably works. If it is equivalent, the honest move is to say so and drop
the claim — and the component itself.

The **single-task** rows answer the question a reader will ask first about any multi-task
model: would two separate models have been better? If the joint model matches or beats both
single-task models, sharing the representation costs nothing and buys a second output from
one forward pass, which is a defensible efficiency claim. If a single-task model wins, the
multi-task framing is a liability rather than a feature and should be reported as such.